In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import tensorflow as tf

# Make data splits and model results reproducible.
tf.keras.utils.set_random_seed(42)

In [102]:
df = pd.read_csv('quote_dataset.csv')

In [103]:
df.head()

,quote,Author
0,“The world as we have created it is a process ...,Albert Einstein
1,"“It is our choices, Harry, that show what we t...",J.K. Rowling
2,“There are only two ways to live your life. On...,Albert Einstein
3,"“The person, be it gentleman or lady, who has ...",Jane Austen
4,"“Imperfection is beauty, madness is genius and...",Marilyn Monroe


In [104]:
df.shape

(3038, 2)

In [105]:
df['quote'][0]

'“The world as we have created it is a process of our thinking. It cannot be changed without changing our thinking.”'

In [106]:
quotes = df['quote']
quotes.head()

0    “The world as we have created it is a process ...
1    “It is our choices, Harry, that show what we t...
2    “There are only two ways to live your life. On...
3    “The person, be it gentleman or lady, who has ...
4    “Imperfection is beauty, madness is genius and...
Name: quote, dtype: str

**<h3>Data Preprocessing</h3>**

*Making the text lower case and removing commas and puntuations*

In [107]:
quotes = quotes.str.lower()

In [108]:
import string
translator = str.maketrans('','',string.punctuation)
quotes = quotes.apply(lambda x: x.translate(translator))

In [109]:
quotes.head()

0    “the world as we have created it is a process ...
1    “it is our choices harry that show what we tru...
2    “there are only two ways to live your life one...
3    “the person be it gentleman or lady who has no...
4    “imperfection is beauty madness is genius and ...
Name: quote, dtype: str

**Tokenization**

In [110]:
from tensorflow.keras.preprocessing.text import Tokenizer

In [111]:
vocab_size = 10000

tokenizer = Tokenizer(num_words=vocab_size)
tokenizer.fit_on_texts(quotes)

In [112]:
word_index = tokenizer.word_index
print(len(word_index))

8978


In [113]:
sequence = tokenizer.texts_to_sequences(quotes)

In [114]:
for i in range(3):
    print(quotes[i])

“the world as we have created it is a process of our thinking it cannot be changed without changing our thinking”
“it is our choices harry that show what we truly are far more than our abilities”
“there are only two ways to live your life one is as though nothing is a miracle the other is as though everything is a miracle”


In [115]:
for i in range(3):
    print(sequence[i])

[713, 62, 29, 19, 16, 946, 10, 7, 5, 1156, 8, 70, 293, 10, 145, 12, 809, 104, 752, 70, 2461]
[947, 7, 70, 871, 373, 9, 433, 21, 19, 465, 14, 294, 52, 54, 70, 3676]
[1337, 14, 53, 201, 714, 3, 81, 15, 36, 37, 7, 29, 329, 93, 7, 5, 1157, 1, 101, 7, 29, 329, 126, 7, 5, 3677]


**Input and output variable**

In [116]:
X = []
y = []

for seq in sequence:
    for i in range(1, len(seq)):
        input_seq = seq[:i]
        output_seq = seq[i]
        X.append(input_seq)
        y.append(output_seq)

In [117]:
len(X), len(y)

(85271, 85271)

**Padding**

In [118]:
max_len = max(len(x) for x in X)
print(max_len)

745


In [119]:
from tensorflow.keras.preprocessing.sequence import pad_sequences

In [120]:
X_padded = pad_sequences(X, maxlen=max_len, padding='pre')

In [121]:
y = np.array(y)

In [122]:
X_padded.shape, y.shape

((85271, 745), (85271,))

**One hot encoding**

In [123]:
# Keep labels as integer class IDs to avoid allocating a dense one-hot matrix.
y_one_hot = np.asarray(y, dtype=np.int32)
print(y_one_hot.shape, y_one_hot.min(), y_one_hot.max())

# Compile the model with loss="sparse_categorical_crossentropy".

(85271,) 1 8978


**Basic RNN model**

In [124]:
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Embedding, LSTM, Dense, SimpleRNN

In [125]:
embedding_dim = 50
rnn_units = 128

In [126]:
from tensorflow.keras import Input


rnn_model = Sequential([
    Input(shape=(max_len,)),
    Embedding(vocab_size, embedding_dim, mask_zero=True),
    SimpleRNN(rnn_units),
    Dense(vocab_size, activation="softmax")
])

In [127]:
rnn_model.compile(optimizer='adam', loss='sparse_categorical_crossentropy', metrics=['accuracy'])

In [128]:
rnn_model.summary()

Model: "sequential_6"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ embedding_5 (Embedding)         │ (None, 745, 50)        │       500,000 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ simple_rnn_3 (SimpleRNN)        │ (None, 128)            │        22,912 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_5 (Dense)                 │ (None, 10000)          │     1,290,000 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 1,812,912 (6.92 MB)

 Trainable params: 1,812,912 (6.92 MB)

 Non-trainable params: 0 (0.00 B)

**LSTM Model**

In [129]:
from tensorflow.keras import Input

lstm_model = Sequential([
    Input(shape=(max_len,)),
    Embedding(
        input_dim=vocab_size,
        output_dim=embedding_dim,
        mask_zero=True
    ),
    LSTM(units=rnn_units),
    Dense(units=vocab_size, activation='softmax')
])

In [130]:
lstm_model.compile(optimizer='adam', loss = 'sparse_categorical_crossentropy', metrics=['accuracy'])

In [131]:
lstm_model.summary()

Model: "sequential_7"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ embedding_6 (Embedding)         │ (None, 745, 50)        │       500,000 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ lstm_2 (LSTM)                   │ (None, 128)            │        91,648 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_6 (Dense)                 │ (None, 10000)          │     1,290,000 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 1,881,648 (7.18 MB)

 Trainable params: 1,881,648 (7.18 MB)

 Non-trainable params: 0 (0.00 B)

**Training RNN**

In [132]:
history_rnn = rnn_model.fit(
    X_padded,
    y_one_hot,
    validation_split=0.2,
    epochs=1,
    batch_size=128,
    verbose=1
)

533/533 ━━━━━━━━━━━━━━━━━━━━ 392s 732ms/step - accuracy: 0.0380 - loss: 6.9091 - val_accuracy: 0.0432 - val_loss: 6.8500


**Training LSTM**

In [ ]:
history_lstm = lstm_model.fit(
    X_padded,
    y_one_hot,
    validation_split=0.2,
    epochs=10,
    batch_size=64
)